In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import requests, re
import time
import json
import os
from xml.etree import ElementTree as ET

## Paris Match

Récupération de l'ensemble des sitemaps depuis 2005

In [ ]:
r = requests.get("http://web.archive.org/cdx/search/cdx?url=parismatch.com/sitemap/*&output=json&collapse=urlkey&from=2000&to=2023")

# Sauvegarde du contenu brut dans un fichier
with open("/content/drive/MyDrive/Colab Notebooks/stage-mids/paris_match.json", "w", encoding="utf-8") as f:
    f.write(r.text)

In [ ]:
with open("/content/drive/MyDrive/Colab Notebooks/stage-mids/paris_match.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# On conserve seulement ceux du type sitemap/année-moi.xml
motif = re.compile(r"https://www.parismatch.com/sitemap/\d{4}-\d{2}\.xml$")

# La ligne [1] contient le timestamp, la [2] l'URL
res = [(lignes[1],lignes[2]) for lignes in data if motif.search(lignes[2])]

# Transformation en URLs Wayback directement utilisables
res = [f"https://web.archive.org/web/{timestamp}id_/{url}" for (timestamp, url) in res]

On passe maintenant au téléchargment de tous les URL

In [ ]:
# On crée un doc qui va stocker les sitemap traitées, et les URL

with open("/content/drive/MyDrive/Colab Notebooks/stage-mids/paris_match_articles.json", "w", encoding="utf-8") as f:
    json.dump({"sitemap": [], "URL": []}, f)

In [ ]:
with open("/content/drive/MyDrive/Colab Notebooks/stage-mids/paris_match_articles.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for page in res:
    if page in data["sitemap"]:
        continue

    try:
        xml = requests.get(page, timeout=30).content
    except:
        continue

    root = ET.fromstring(xml)
    articles_url = root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")

    data["URL"].extend(url.text for url in articles_url)
    data["sitemap"].append(page)

    with open("/content/drive/MyDrive/Colab Notebooks/stage-mids/paris_match_articles.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)

    time.sleep(15)

Pour les autres journaux, on va créer un dictionnaire qui contiennent les sitemap pour chaque site. On traitera à part Les Echos, qui permettent de télécharger des zip, qui contienne les xml, et Libération, dont on a pas accés trouvé une carte satisfaisant pour l'instant. VA est à étudié séparément aussi. Et Télérama séparément car le site est en php plutôt que en XML. Et le nouvel obs, il faut sauter le premier lien, ce n'est pas un URL.Pareil pour Nice Matin.


In [ ]:
journaux = {
    "le figaro" : "https://sitemaps.lefigaro.fr/lefigaro.fr/articles.xml",
    "le capital" : "https://www.capital.fr/sitemap/articles.xml",
    "nice matin" : "https://www.nicematin.com/sitemap.xml",
    "le nouvel observateur" : "https://www.nouvelobs.com/sitemap/sitemap-index-articles.xml"
}

### Le Journal du Dimanche

In [ ]:
nom_journal = "le journal du dimanche"
url_index = "https://www.lejdd.fr/sitemap.xml"
FICHIER = f"/content/drive/MyDrive/Colab Notebooks/stage-mids/{nom_journal.replace(' ', '_')}_articles.json"

# Première rubrique n'est pas un sous-sitemap
to_jump = [ "https://www.nicematin.com/sitemap_fixed.xml", "https://www.nouvelobs.com/sitemap/sitemap-index-categories.xml"]
echecs = []

# On stock les résultats dans data
data = {"sitemap": [], "URL": []}

# Étape 1 : récupérer la liste des sous-sitemaps
xml = requests.get(url_index, timeout=30).content
root = ET.fromstring(xml)
sous_sitemaps = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]

# Étape 2 : parcourir chaque sous-sitemap
for page in sous_sitemaps:
    if page in to_jump:
        continue

    try:
        xml = requests.get(page, timeout=30).content
    except:
        echecs.append(page)
        continue

    if not xml:
        echecs.append(page)
        continue

    root = ET.fromstring(xml)
    articles_url = root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")

    data["URL"].extend(url.text for url in articles_url)
    data["sitemap"].append(page)

    with open(FICHIER, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)

    time.sleep(1)

## Le Monde

In [ ]:
import os, json, time, requests
from xml.etree import ElementTree as ET

nom_journal = "le monde"
url_index = "https://www.lemonde.fr/sitemap_index.xml"
FICHIER = f"/content/drive/MyDrive/Colab Notebooks/stage-mids/{nom_journal.replace(' ', '_')}_articles.jsonl"


# Reconstruire l'état depuis le fichier
deja_traites = set()
nb_urls = 0
nb_echecs = 0

if os.path.exists(FICHIER):
    with open(FICHIER, "r", encoding="utf-8") as f:
        for ligne in f:
            entree = json.loads(ligne)
            deja_traites.add(entree["sitemap"])
            if entree["type"] == "ok":
                nb_urls += len(entree["urls"])
            else:
                nb_echecs += 1


# Récupérer la liste des sous-sitemaps
xml = requests.get(url_index, timeout=30).content
root = ET.fromstring(xml)
sous_sitemaps = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]

print(f"{len(sous_sitemaps)} sous-sitemaps à traiter au total")

# Boucle principale
for i, page in enumerate(sous_sitemaps):

    if page in deja_traites:
        continue

    try:
        xml = requests.get(page, timeout=30).content
        if not xml:
            raise ValueError("contenu vide")
        root = ET.fromstring(xml)
    except:
        with open(FICHIER, "a", encoding="utf-8") as f:
            f.write(json.dumps({"type": "echec", "sitemap": page}) + "\n")
        nb_echecs += 1
        deja_traites.add(page)
        continue

    articles_url = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]

    with open(FICHIER, "a", encoding="utf-8") as f:
        f.write(json.dumps({"type": "ok", "sitemap": page, "urls": articles_url}) + "\n")

    nb_urls += len(articles_url)
    deja_traites.add(page)

    if i % 100 == 0:
        print(f"[{i}/{len(sous_sitemaps)}]  Sitemaps: {len(deja_traites)}  URLs: {nb_urls}  Echecs: {nb_echecs}")

    time.sleep(1)

3142 sous-sitemaps à traiter au total
[0/3142]  Sitemaps: 1  URLs: 195  Echecs: 0
[100/3142]  Sitemaps: 101  URLs: 68541  Echecs: 0
[200/3142]  Sitemaps: 201  URLs: 146554  Echecs: 0
[300/3142]  Sitemaps: 301  URLs: 259784  Echecs: 4
[400/3142]  Sitemaps: 401  URLs: 389840  Echecs: 4
[500/3142]  Sitemaps: 501  URLs: 485540  Echecs: 4
[600/3142]  Sitemaps: 601  URLs: 556779  Echecs: 4
[700/3142]  Sitemaps: 701  URLs: 627011  Echecs: 4
[800/3142]  Sitemaps: 801  URLs: 698572  Echecs: 4
[900/3142]  Sitemaps: 901  URLs: 807394  Echecs: 4
[1000/3142]  Sitemaps: 1001  URLs: 809260  Echecs: 4
[1100/3142]  Sitemaps: 1101  URLs: 810349  Echecs: 4
[1200/3142]  Sitemaps: 1201  URLs: 811167  Echecs: 4
[1300/3142]  Sitemaps: 1301  URLs: 813747  Echecs: 4
[1400/3142]  Sitemaps: 1401  URLs: 818550  Echecs: 4
[1500/3142]  Sitemaps: 1501  URLs: 819415  Echecs: 4
[1600/3142]  Sitemaps: 1601  URLs: 820500  Echecs: 4
[1700/3142]  Sitemaps: 1701  URLs: 822364  Echecs: 4
[1800/3142]  Sitemaps: 1801  URLs: 9

Test 1

In [ ]:
echecs = []
with open(FICHIER, "r", encoding="utf-8") as f:
    for ligne in f:
        entree = json.loads(ligne)
        if entree["type"] == "echec":
            echecs.append(entree["sitemap"])

print(f"{len(echecs)} sitemaps en échec")
for url in echecs[:10]:
    print(" ", url)

9 sitemaps en échec
  https://www.lemonde.fr/sitemap/articles/2005-08-01.xml
  https://www.lemonde.fr/sitemap/articles/2006-06-05.xml
  https://www.lemonde.fr/sitemap/articles/2007-02-19.xml
  https://www.lemonde.fr/sitemap/articles/2008-04-21.xml
  https://www.lemonde.fr/sitemap/year/articles/2006-06-01.xml
  https://www.lemonde.fr/sitemap/year/articles/2007-02-01.xml
  https://www.lemonde.fr/sitemap/year/articles/2007-04-01.xml
  https://www.lemonde.fr/sitemap/year/articles/2007-05-01.xml
  https://www.lemonde.fr/sitemap/year/articles/2007-11-01.xml


In [3]:
journaux = {"le journal du dimanche": "https://www.lejdd.fr/sitemap.xml"}

Pour les journaux restants

In [4]:
for journal, url in journaux.items():
  nom_journal = journal
  url_index = url
  FICHIER = f"/content/drive/MyDrive/Colab Notebooks/stage-mids/{nom_journal.replace(' ', '_')}_articles.jsonl"


  # Reconstruire l'état depuis le fichier
  deja_traites = set()
  nb_urls = 0
  nb_echecs = 0
  to_jump = [ "https://www.nicematin.com/sitemap_fixed.xml", "https://www.nouvelobs.com/sitemap/sitemap-index-categories.xml"]

  if os.path.exists(FICHIER):
      with open(FICHIER, "r", encoding="utf-8") as f:
          for ligne in f:
              entree = json.loads(ligne)
              deja_traites.add(entree["sitemap"])
              if entree["type"] == "ok":
                  nb_urls += len(entree["urls"])
              else:
                  nb_echecs += 1


  # Récupérer la liste des sous-sitemaps
  xml = requests.get(url_index, timeout=30).content
  root = ET.fromstring(xml)
  sous_sitemaps = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]

  print(f"{len(sous_sitemaps)} sous-sitemaps à traiter au total")

  # Boucle principale
  for i, page in enumerate(sous_sitemaps):

      if page in to_jump:
        continue

      if page in deja_traites:
          continue

      try:
          xml = requests.get(page, timeout=30).content
          if not xml:
              raise ValueError("contenu vide")
          root = ET.fromstring(xml)
      except:
          with open(FICHIER, "a", encoding="utf-8") as f:
              f.write(json.dumps({"type": "echec", "sitemap": page}) + "\n")
          nb_echecs += 1
          deja_traites.add(page)
          continue

      articles_url = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]

      with open(FICHIER, "a", encoding="utf-8") as f:
          f.write(json.dumps({"type": "ok", "sitemap": page, "urls": articles_url}) + "\n")

      nb_urls += len(articles_url)
      deja_traites.add(page)

      if i % 100 == 0:
          print(f"[{i}/{len(sous_sitemaps)}]  Sitemaps: {len(deja_traites)}  URLs: {nb_urls}  Echecs: {nb_echecs}")

      time.sleep(1)

16 sous-sitemaps à traiter au total
[0/16]  Sitemaps: 1  URLs: 9225  Echecs: 0


In [7]:
fichier = DOSSIER / "le_journal_du_dimanche_articles.jsonl"

if fichier.exists():
    nb_sitemaps_ok = 0
    nb_sitemaps_echec = 0
    nb_urls = 0

    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            entree = json.loads(ligne)
            if entree["type"] == "ok":
                nb_sitemaps_ok += 1
                nb_urls += len(entree["urls"])
            else:
                nb_sitemaps_echec += 1

    print(f"{fichier.name}")
    print(f"  Sitemaps OK     : {nb_sitemaps_ok}")
    print(f"  Sitemaps échec  : {nb_sitemaps_echec}")
    print(f"  URLs collectées : {nb_urls}")

le_journal_du_dimanche_articles.jsonl
  Sitemaps OK     : 16
  Sitemaps échec  : 0
  URLs collectées : 159225


In [ ]:
# Compter Paris Match (ancien format)
fichier_pm = DOSSIER / "le_journal_du_dimanche_articles.json"
if fichier_pm.exists():
    with open(fichier_pm, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"  Sitemaps OK     : {len(data.get('sitemap', []))}")
    print(f"  Sitemaps échec  : {len(data.get('echecs', []))}")
    print(f"  URLs collectées : {len(data.get('URL', []))}")

  Sitemaps OK     : 15
  Sitemaps échec  : 0
  URLs collectées : 145571


Transformation de JSON à JSONL pour Paris Match (trop compliqué de refaire le scrapping via wayback)

In [2]:
from pathlib import Path

DOSSIER = Path("/content/drive/MyDrive/Colab Notebooks/stage-mids/")

source = DOSSIER / "paris_match_articles.json"
dest = DOSSIER / "paris_match_articles.jsonl"

with open(source, "r", encoding="utf-8") as f:
    data = json.load(f)

sitemaps_ok = data.get("sitemap", [])
urls_toutes = data.get("URL", [])
echecs = data.get("echecs", [])

with open(dest, "w", encoding="utf-8") as f:
    for i, sitemap in enumerate(sitemaps_ok):
        if i == 0:
            ligne = {"type": "ok", "sitemap": sitemap, "urls": urls_toutes}
        else:
            ligne = {"type": "ok", "sitemap": sitemap, "urls": []}
        f.write(json.dumps(ligne, ensure_ascii=False) + "\n")

    for sitemap in echecs:
        ligne = {"type": "echec", "sitemap": sitemap}
        f.write(json.dumps(ligne, ensure_ascii=False) + "\n")

print(f"Converti: {source.name} → {dest.name}")
print(f"  Sitemaps OK     : {len(sitemaps_ok)}")
print(f"  Sitemaps échec  : {len(echecs)}")
print(f"  URLs collectées : {len(urls_toutes)}")

Converti: paris_match_articles.json → paris_match_articles.jsonl
  Sitemaps OK     : 203
  Sitemaps échec  : 0
  URLs collectées : 213509
